# **SAE Lens: Basics of SAE in Python**


SAE Lens is a relatively new framework devoted entirely to SAEs. The field is only beginning to develop, so the framework does not offer much yet, and the library changes quickly.

In this practice you will reach two goals:


1.   Understand what an SAE looks like in Python;
2.   Reproduce the workflow for analyzing the features an SAE has learned.

Good luck!


## **Set Up**
Preparing the environment. When choosing a runtime, pick CPU: this practice does not need extra compute.


In [ ]:
!pip install transformer-lens==3.9.0 plotly -q

In [ ]:
!pip install sae-lens==6.54.0 -q

****
**After running the cells above, restart the runtime before starting the tasks.**
****

In [ ]:
COLAB = True

# Standard libraries
import os
import torch
from tqdm import tqdm
import plotly.express as px
import pandas as pd

# To save time
#torch.set_grad_enabled(False)


if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

**Make sure the `sae_lens` library version is >= 6.0.0.**

In [ ]:
!pip show sae-lens

# **Working with autoencoders**

Autoencoders are models, so you work with them the same way as with any other model.

For example, when you need to study some model, you can first look for an SAE *already trained on the model you are interested in*. You can do this either by searching the models on [Hugging Face](https://huggingface.co/models?search=SAE) or by using the list of trained SAEs in SAE Lens.


**Question. How many different SAEs can be trained on one model?**

`Choose the answer in the lesson`

In [ ]:
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

df = pd.DataFrame.from_records(
    {k: v.__dict__ for k, v in get_pretrained_saes_directory().items()}
).T.reset_index(drop=True)

df.head()

The resulting table is organized as follows:

- `release` — a release that reflects a particular approach to SAEs. For example:
  - gemma-scope-2b-pt-att
    - gemma-scope- — SAEs belonging to the [Gemma-Scope](https://huggingface.co/google/gemma-scope) project
    - 2b-pt- — the SAEs are trained on the base model Gemma v2 2B
    - att:  the SAEs are trained on the outputs of the attention layer

- `repo_id` — the Hugging Face repository where the models are stored. It often has a thorough model card too, which is always worth reading.
- `model` — the model the autoencoder belongs to
- `saes_map` — the list of all SAEs in this release
- `neuronpedia_id` — filled in if the SAE is on Neuronpedia
- `conversion_func`, `expected_var_explained`, `expected_l0`, `config_overrides` — service columns.

There is no gold standard for naming SAEs in the library yet, so the table can look chaotic. Still, the example of good, informative names below should help you when you load your own models.

In [ ]:
print("SAEs in the GPT2 Small Resid Pre release")
for k, v in df.loc[df.release == "gpt2-small-res-jb", "saes_map"].values[0].items():
    print(f"SAE id: {k} for hook point: {v}")

print("-" * 50)
print("SAEs in the feature splitting release")
for k, v in (
    df.loc[df.release == "gpt2-small-res-jb-feature-splitting", "saes_map"]
    .values[0]
    .items()
):
    print(f"SAE id: {k} for hook point: {v}")

print("-" * 50)
print("SAEs in the Gemma base model release")
for k, v in df.loc[df.release == "gemma-2b-res-jb", "saes_map"].values[0].items():
    print(f"SAE id: {k} for hook point: {v}")

# **Working with an SAE-model pair**

Let us load an SAE-model pair and look a little more at the naming conventions, namely the `pre`, `post`, `attn`, `resid` suffixes in the code.


In [ ]:
from sae_lens import SAE, HookedSAETransformer
from transformers import GPT2Tokenizer, GPT2Model
import torch

# For comparison with the hooked model, also load just a small GPT fine-tuned on Portuguese
hooked_model = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
basic_model = GPT2Model.from_pretrained("pierreguillou/gpt2-small-portuguese").to(device)

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # release
    sae_id="blocks.7.hook_resid_pre",  # SAE id
    device=device,

)

sae_model = sae  # in sae-lens 6 SAE.from_pretrained returns only the SAE, without a tuple
sae_cfg = sae.cfg.to_dict()  # the SAE config as a dictionary

Above we used an SAE from block 7 (`blocks.7`), trained with a hook on the residual stream for the tokens $x_{pre}$ (`hook_resid_pre`). Let us find this point in the model. First, recall how an embedding is transformed as it passes through a block.

`x_pre ──► Attention ──► x_mid ──► MLP ──► x_post (= x_pre of next layer)`

Now let us look at block 7 in the code of the base and the hooked GPT2.

In [ ]:
hooked_model.blocks[7]

In [ ]:
basic_model.h[7]

From the hook names you can tell which point of the block's output each of them stores.

**Task. Match each hook with its point in the model in the lesson.**

1. hook_normalized
2. hook_q_input
3. hook_resid_mid
4. hook_resid_post

- i. The input embedding before multiplication by the matrix $Q$;
- ii. The representation after attention and before the MLP;
- iii. The block output after all transformations;
- iv. The output after layernorm.

### **SAE**
Now let us see what an SAE looks like.

In [ ]:
sae_model # the model, also covered with hooks

In [ ]:
sae_cfg # the full model config

In [ ]:
sae_model.cfg.__dict__ # the full model config in a more compact form

<details>
<summary><b>Footnote describing the parameters</b></summary>

1. `architecture`: Defines the type of SAE architecture. Here it is the standard encoder-decoder architecture with intermediate activations.
2. `d_in`: The input dimension of the SAE, here 768.
3. `d_sae`: The size of the SAE hidden layer, here 24576. It equals the number of possible feature activations.
5. `apply_b_dec_to_input`: Defines whether the decoder bias is applied to the input.
7. `context_size`: The size of the context window, here 128 tokens. It turned out that on prompts longer than the training context such SAEs [perform worse](https://www.lesswrong.com/posts/8QRH8wKcnKGhpAu2o/examining-language-model-performance-with-reconstructed) (Anders, Bloom).
8. `model_name`: The name of the model used, here `'gpt2-small'`. [It is a valid model name in TransformerLens](https://transformerlensorg.github.io/TransformerLens/generated/model_properties_table.html).
9. `hook_name`: The specific hook in the model that the SAE is attached to.
10. `hook_head_index`: Defines which attention head to attach to. Not used here, since this SAE works with the residual stream.
11. `prepend_bos`: Whether to add the beginning-of-sequence token at the start. Set to `True` here.
12. `dataset_path`: The path to the dataset used for training or evaluation. It can be local or from HuggingFace.
14. `normalize_activations`: The method for normalizing activations. In this configuration it is `'none'`, so no normalization is applied.
15. `dtype`: The data type for tensor operations. Here it is a 32-bit floating-point number.
16. `device`: The compute device (for example, CPU or GPU).
17. `sae_lens_training_version`: The SAE Lens version used for training.
18. `activation_fn_kwargs`: Extra parameters for the activation function. For example, if the `topk` function is chosen, the value of `k` goes here.
19. `neuronpedia_id` — the id on Neuronpedia. If it is filled in, the SAE has an interactive visualization on that [site](https://www.neuronpedia.org/).  
</details>

### **Features in an SAE, visualization with Neuronpedia**


#### **Random features**


Using the convenient Neuronpedia integration, let us look at the features in an SAE. Recall that, ideally, an SAE should give a sparse representation for every input example, that is, activate $n$ features with $n$ much smaller than $d_{sae}$.

Each feature is a neuron. Neurons are labeled as features empirically.

A hypothetical example: if, over repeated runs on test data, the neuron $f$ with index $x$ ($f_x$) always activates on words related to cats, we assign it the feature “phrases related to cats”.

Now let us look at a real example.

In [ ]:
from IPython.display import IFrame

# Look at a feature with a random index
feature_idx = 3777

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"


def get_dashboard_html(sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)


html = get_dashboard_html(
    sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=feature_idx
)
IFrame(html, width=1200, height=600)

Features in each SAE were labeled either by a person or by an explainer model: feature 3777 on Neuronpedia has two descriptions, both from explainer models (gpt-3.5-turbo and gpt-4o-mini). The components of the resulting dashboard let you analyze:

1. The feature description (top left, Feature Description) — *phrases related to anticipation or holding back*.

2. Logit plots (left, Logit Plots) — the vocabulary tokens whose logits the feature's decoder direction raises (positive) and lowers (negative) the most.

3. Density plots (right, Activations Density Plot):
 - the upper plot is a histogram of the feature's non-zero activations on a random sample of texts: the x axis is the strength of the feature activation, the y axis is the number of tokens;
 - the lower plot is a histogram of the feature's contributions to the logits of all GPT-2 vocabulary tokens: negative values belong to tokens the feature suppresses, positive values to tokens it promotes.

4. A field for testing the feature activation — Test Activation. Here you can enter your own text and see how the feature responds to it.

5. Examples with the strongest activations of the feature — Top Activations, below the text. On them you can analyze the sentences and the place where the feature fires.

**Question. Based on the plots, choose the true statements in the lesson:**

- The lower density plot (the feature's contributions to the token logits) is unimodal: a peak near zero, decaying on both sides. In shape it is close to a normal distribution;
- The largest negative contribution to the logits in absolute value (Negative Logits) is greater than the largest positive one (Positive Logits);
- Most often, the nonzero activations of the feature take values no greater than 5: the tallest bars of the upper plot are at the start of the axis;
- The first three bars of the upper plot lie to the left of activation 3;
- All three first bars of the upper plot lie to the left of activation 1

#### **Searching for features on a given topic**


For some models SAE Lens has feature labels obtained by annotation with another model. Using these labels, you can try to search for features on a given topic.

Let us do this with the features of the Gemma model.

In [ ]:
import requests
import gzip
import json
import re

# The Neuronpedia /api/explanation/export endpoint has been removed. Feature explanation exports
# live in the public neuronpedia-datasets S3 bucket: v1/<model>/<SAE>/explanations/*.jsonl.gz
def load_explanations(model_id, sae_id):
    base = "https://neuronpedia-datasets.s3.us-east-1.amazonaws.com"
    listing = requests.get(base, params={"list-type": "2", "prefix": f"v1/{model_id}/{sae_id}/explanations/"}).text
    rows = []
    for key in re.findall(r"<Key>([^<]+\.jsonl\.gz)</Key>", listing):
        raw = gzip.decompress(requests.get(f"{base}/{key}").content).decode()
        rows += [{k: v for k, v in json.loads(line).items() if k != "embedding"} for line in raw.splitlines()]
    return rows


#data = load_explanations("gpt2-small", "7-res-jb")
data = load_explanations("gemma-2-2b", "20-gemmascope-res-16k")

In [ ]:
# convert to pandas
explanations_df = pd.DataFrame(data)

# rename index to "feature"
explanations_df.rename(columns={"index": "feature"}, inplace=True)

explanations_df["description"] = explanations_df["description"].apply(
    lambda x: x.lower()
)
explanations_df.head()

**Question. The SAE `20-gemmascope-res-16k` of gemma-2-2b has 16384 features, numbered 0 to 16383. How many of them have no description at all in the downloaded export?**

Hint: the export has one row per description, and the feature number is in the `feature` column.

In [ ]:
 # Your code here

Let us look at the neurons that encode features related to fruit.

The gemma table may come out empty: the features of this SAE on Neuronpedia are now relabeled by a single model, and none of their descriptions contains the word fruits. For GPT-2 below such descriptions exist.

In [ ]:
fruits_features = explanations_df.loc[explanations_df.description.str.contains(" fruits")]
fruits_features

**Do the same for the SAE `7-res-jb` of the GPT2 model. Get the table and find the features whose description contains the word fruits. How many SAE features did you get?**

In [ ]:
 # Your code here

In [ ]:
fruits_features = explanations_df.loc[explanations_df.description.str.contains(" fruits")]
fruits_features

In [ ]:
# After finding the features, run the cell to see the dashboard for one of them
html = get_dashboard_html(
    sae_release="gpt2-small",
    sae_id="7-res-jb",
    feature_idx=fruits_features.feature.values[0],
)
IFrame(html, width=1200, height=600)

### **Getting features with an SAE**

As the last step, let us see how to extract features using a trained SAE with labeled (or unlabeled) features.


In [ ]:
from transformer_lens.utils import test_prompt

prompt = "The best animal of the world is a"
answer = "cat"

# See how the model predicts the end of the sentence
test_prompt(prompt, answer, hooked_model)

### **Running the model with an SAE**

Finally, let us see what a run of the model with an SAE can look like. Recall what the SAE class looks like in SAE Lens.


In [ ]:
sae

Note that the SAE has hooks too. They let you analyze passes at different points of the SAE as well, which is very useful for interpretation. Namely:

- `hook_sae_input` — the input ($input$) to the SAE, that is, the embedding from the model fed into the autoencoder. Shape: `[batch, seq_len, d_model]`;

- `hook_sae_acts_pre` — the sparse representation before ReLU, the result of multiplying the input by the encoder weight matrix.
Formally, with `apply_b_dec_to_input=True`, as in this SAE: $$acts_{pre} = (x - b_{dec})W_{enc} + b_{enc}$$ Shape: `[batch, seq_len, d_sae]`;

- `hook_sae_acts_post` — the sparse representation after ReLU. This is what is usually taken as the interpretable representation. Formally: $$acts_{post} = ReLU(acts_{pre})$$ Shape: `[batch, seq_len, d_sae]`;


- `hook_sae_recons` — the reconstruction after the SAE, but before inverse normalization and reshape, if they are applied. The “raw” reconstructed activations. Formally:
$$acts_{post}W_{dec} + b_{dec}$$ Shape: `[batch, seq_len, d_model]`;


- `hook_sae_error` — the difference between the reconstruction and the original activation, that is, formally: $$error = input - reconstruction$$ Shape: `[batch, seq_len, d_model]`;

- `hook_sae_output` — the final reconstructed activations after all steps (inverse normalization, reshaping). With use_error_term=True the reconstruction error (hook_sae_error) is added here.
Formally: $$recons = acts_{post}W_{dec} + b_{dec} + error$$ Shape: `[batch, seq_len, d_model]`;



**The `use_error_term` parameter**.

When the model is run with an SAE, the next layer after the SAE receives the autoencoder's reconstructions as input. Ideally (if the autoencoder is trained perfectly) this should not affect the model's behavior, but in practice SAEs do not reconstruct activations exactly.  So, to keep the model's performance, the error $error$ is added to the reconstruction to get the same embedding and the same performance.

In SAE Lens this logic is implemented by the `sae.use_error_term` attribute. If you set it to `True`, the reconstruction after the SAE will take the approximation error into account.

In [ ]:
sae.use_error_term  # With the error term, the original embedding is passed further through the model. Leave False

In [ ]:
# Get the SAE representations by running the model with hooks and the SAE
_, cache = hooked_model.run_with_cache_with_saes(prompt, saes=[sae])

k, v = list(cache.items())[0]
print(k, v.shape)

Each collected value in the layer is a representation from the model across all hooks. Analyze the cache and check how many representations are related to the SAE.

In [ ]:
for k, v in cache.items():
  if 'sae' in k:
    print(k, v.shape)

There are fewer representations than hooks in the SAE. This is because of the `use_error_term` parameter: what was not needed was not computed and, accordingly, not recorded.

Now let us see which features activated on the seventh layer and what the representation looked like.

In [ ]:
px.line(
    cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0, -1, :].cpu().numpy(),
    title="Feature activations at the final token position",
    labels={"index": "Feature", "value": "Activation"},
).show()

# Top 5 tokens
vals, inds = torch.topk(
    cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0, -1, :], 5
)
for val, ind in zip(vals, inds):
    print(f"Feature {ind} fired {val:.2f}")
    html = get_dashboard_html(
        sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=ind
    )
    display(IFrame(html, width=1200, height=300))

Not a single feature related to cats. And this is not anomalous behavior. Recall the lesson “Logit Lens” of the module “Mechanistic interpretability”: models build their answer gradually, processing the input layer by layer.

## **Conclusion**

In this practice you did some basic work with trained SAEs and analyzed the range of tasks a trained autoencoder can solve. It is worth highlighting the following details:

- SAE Lens provides a convenient wrapper for working with SAEs. But SAEs are still standalone models, and you can write your own configuration for your needs, including on top of the library's convenient codebase.
- Analyzing the features of an already trained SAE requires labeling. It can be automated; we will discuss the methods later in practice and theory.
- When analyzing the learned features, you can use different scenarios: analyze slightly changed prompts to see how the activation changes, look at the examples and tokens where a feature's activation is highest. Besides, an SAE can be used for steering.

A trained SAE is the start of a big field for creativity. Study and explore!